# Hidden Value: Points Projection (companion notebook)

**Goal:** project how many points each team scores in next week's games, from the drive table.

This is a companion to the drive model, not part of it. The research question in the design doc is about 2026 drives. This notebook is a forecasting tool, so its constants were set on **2021-2025** and are never tuned on 2026 results. That keeps the two from contaminating each other.

**How a projection is built**
1. Offense rating = points per drive (7/3/0 convention), with drives run by the expected starting QB counted fully and drives run by any other QB counted at 25%.
2. Defense rating = points per drive allowed.
3. Both are pulled toward league average by adding 20 league-average drives. A team with few drives gets pulled harder, so **no drive gate is needed here**.
4. Points = a fixed formula on those two ratings, home field, and how much of the season the expected starter has actually played.

**What the backtest says (2021-2025, weeks 4-18, 2,238 team-games, each season predicted from the other four)**

| Method | Avg miss per team (pts) | Picks winner |
|---|---|---|
| League average for everyone | 7.96 | n/a |
| Points for / points against | 7.59 | 62% |
| Drive model, no QB information | 7.57 | 60% |
| **This notebook (drive model + QB information)** | **7.50** | **62%** |
| Betting market | 7.25 | 68% |

Read that honestly: a single team's score is mostly noise, the market is better than this model, and the drive stats alone are no better than plain points for and against. The part that adds something is knowing who is at quarterback: without it, teams starting a QB who had run under 30% of their drives were over-projected by 2.2 points on average, and with it that bias is about zero.

Drives used here are **all non-clock drives, garbage time included**. In the backtest the garbage-time filter made projections slightly worse, not better (less data). The filter stays in the research analysis; it is just not used for forecasting.

## 1. Setup and load

In [34]:
import nflreadpy as nfl
import pandas as pd, numpy as np
import os, glob

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

SEASON = 2026
FOLDER = r"C:\Users\brade\OneDrive - Worcester Polytechnic Institute (wpi.edu)\Desktop\Football"
CSV = os.path.join(FOLDER, "hidden_value_2026_drives.csv")

def to_pd(x, cols=None):
    """nflreadpy returns Polars; convert to pandas. cols keeps memory down on big pulls."""
    if cols is not None and hasattr(x, "select"):
        x = x.select([c for c in cols if c in x.columns])
    try:
        return x.to_pandas()
    except AttributeError:
        return x if cols is None else x[[c for c in cols if c in x.columns]]

drives = pd.read_csv(CSV)

PBP_COLS = ["game_id", "season", "season_type", "week", "posteam", "defteam", "fixed_drive",
            "fixed_drive_result", "play_type", "qb_kneel", "qb_spike", "qb_dropback", "id", "name", "play_id"]
pbp = to_pd(nfl.load_pbp(seasons=[SEASON]), PBP_COLS)
pbp = pbp[pbp["season_type"] == "REG"].copy()

sched = to_pd(nfl.load_schedules(seasons=[SEASON]))
sched = sched[sched["game_type"] == "REG"].copy()
NEUTRAL_SITE_GAMES = ["2026_05_PHI_JAX"]   # Eagles vs Jaguars, London
sched.loc[sched["game_id"].isin(NEUTRAL_SITE_GAMES), "location"] = "Neutral"

# The drive CSV must contain every game that has been played. A stale CSV silently
# drops whole games from every rating below.
# Week being projected = the next week with unplayed games. Set WEEK by hand to project a different one.
WEEK = int(sched.loc[sched["home_score"].isna(), "week"].min())
print("projecting week", WEEK, "using drives from weeks before it")

csv_games, pbp_games = set(drives["game_id"]), set(pbp["game_id"])
print("drives:", len(drives), "| games in CSV:", len(csv_games), "| games in play-by-play:", len(pbp_games))
missing = sorted(pbp_games - csv_games)
if missing:
    print(">> STALE CSV: re-run Phase 2 first. Games missing from the drive table:", missing)
else:
    print("drive table is current through week", int(drives["week"].max()))

projecting week 9 using drives from weeks before it
drives: 2559 | games in CSV: 121 | games in play-by-play: 135
>> STALE CSV: re-run Phase 2 first. Games missing from the drive table: ['2026_09_ARI_SEA', '2026_09_CIN_ATL', '2026_09_CLE_NO', '2026_09_DAL_IND', '2026_09_DEN_CAR', '2026_09_DET_MIA', '2026_09_GB_NE', '2026_09_HOU_LAC', '2026_09_JAX_BAL', '2026_09_LA_WAS', '2026_09_LV_SF', '2026_09_NYG_PHI', '2026_09_NYJ_KC', '2026_09_TB_CHI']


## 2. Constants (set on 2021-2025, do not tune on 2026)

Section 8 at the bottom re-derives every number in this cell. If you change one, change it because the backtest says so, and log it.

In [35]:
W_OTHER_QB = 0.25   # weight on drives run by a QB other than the expected starter
K_OFF = 20          # league-average drives added to every offense rating
K_DEF = 20          # league-average drives added to every defense rating

# points = const + off*offense_rating + deff*defense_rating_of_opponent + home*(+1 home, -1 away, 0 neutral)
#          + new_qb*(share of the team's drives the expected starter has NOT run)
COEF = {"const": 4.41, "off": 6.22, "deff": 2.47, "home": 1.15, "new_qb": -3.38}

TYPICAL_MISS = 7.5  # average absolute miss per team, in points, from the backtest

## 3. Who ran each drive

One QB per drive: whoever took the most dropbacks on it. A drive with no dropbacks takes the QB from the team's nearest drive in that game.

In [36]:
def drive_qb(pbp):
    p = pbp[pbp["posteam"].notna() & pbp["fixed_drive"].notna()].copy()
    p["drive_id"] = p["game_id"].astype(str) + "_" + p["fixed_drive"].astype(int).astype(str)
    db = p[(p["qb_dropback"] == 1) & p["id"].notna()]
    top = (db.groupby(["drive_id", "id", "name"]).size().rename("n").reset_index()
             .sort_values(["n", "id"]).drop_duplicates("drive_id", keep="last"))
    return top.set_index("drive_id")[["id", "name"]].rename(columns={"id": "qb_id", "name": "qb"})

d = drives.drop(columns=[c for c in ["qb_id", "qb"] if c in drives.columns]).join(drive_qb(pbp), on="drive_id")
d = d.sort_values(["game_id", "posteam", "fixed_drive"])
for c in ["qb_id", "qb"]:
    d[c] = d.groupby(["game_id", "posteam"])[c].ffill()
    d[c] = d.groupby(["game_id", "posteam"])[c].bfill()
print("drives with no QB assigned (should be 0):", int(d["qb_id"].isna().sum()))

# forecasting frame: every drive from weeks BEFORE the one being projected, except clock-expiration
# drives (garbage time INCLUDED, see intro). The week filter stops a game from helping to predict itself.
f = d[(d["clock_expiration"] == 0) & (d["week"] < WEEK)].copy()
print("drives in forecasting frame:", len(f), "| fewest on one team:", int(f.groupby("posteam").size().min()))

drives with no QB assigned (should be 0): 0
drives in forecasting frame: 2453 | fewest on one team: 63


## 4. Expected starters

Default = the QB who ran the most drives in each team's most recent game. **Check this table every week** and put changes in `STARTER_OVERRIDES` (team -> QB name exactly as it appears in the `qb` column, e.g. `"WAS": "J.Daniels"`). A QB who has not played for the team yet is allowed; he just gets a share of 0.

The schedule file has QB columns too, but they disagree with the play-by-play in several 2026 games, so they are not used here.

In [37]:
STARTER_OVERRIDES = {
     "BAL": "J.Fagnano",   # Lamar and Huntley on IR
    "CHI": "C.Williams",
    "ATL": "M.Penix",
    "PHI": "J.Milton",    # McKee benched
    "TB": "B.Mayfield",
    "NYG": "J.McCarthy",
    "CLE": "S.Sanders",   # Watson in concussion protocol
    "SEA": "D.Lock",      # Darnold out, quad
    "MIN": "C.Wentz",     # Murray out, hamstring
    "ARI": "G.Minshew",   # Brissett benched, then traded
}

def expected_starters(f, overrides=None):
    last_wk = f.groupby("posteam")["week"].transform("max")
    lg = f[f["week"] == last_wk]
    s = (lg.groupby(["posteam", "qb"]).size().rename("n").reset_index()
           .sort_values(["n", "qb"]).drop_duplicates("posteam", keep="last").set_index("posteam")["qb"])
    for team, qb in (overrides or {}).items():
        if team not in s.index:
            raise ValueError(f"override team {team!r} is not a team in the drive table")
        if qb not in set(f.loc[f["posteam"] == team, "qb"]):
            print(f"   note: {qb} has no drives for {team} this season, so he is treated as a brand-new starter")
        s[team] = qb
    return s

starters = expected_starters(f, STARTER_OVERRIDES)

own = f[f["qb"] == f["posteam"].map(starters)]
tbl = pd.DataFrame({"expected_starter": starters,
                    "his_drives": own.groupby("posteam").size().reindex(starters.index).fillna(0).astype(int),
                    "team_drives": f.groupby("posteam").size(),
                    "last_week_he_played": own.groupby("posteam")["week"].max().reindex(starters.index)})
tbl["qb_share"] = (tbl["his_drives"] / tbl["team_drives"]).round(2)
print("teams where the expected starter has NOT run every drive (these are the ones to double-check):")
print(tbl[tbl["qb_share"] < 0.999].sort_values("qb_share").to_string())

   note: J.Milton has no drives for PHI this season, so he is treated as a brand-new starter
   note: G.Minshew has no drives for ARI this season, so he is treated as a brand-new starter
teams where the expected starter has NOT run every drive (these are the ones to double-check):
        expected_starter  his_drives  team_drives  last_week_he_played  qb_share
posteam                                                                         
ARI            G.Minshew           0           87                  NaN      0.00
PHI             J.Milton           0           81                  NaN      0.00
CLE            S.Sanders           1           84                  8.0      0.01
BAL            J.Fagnano           8           85                  8.0      0.09
SEA               D.Lock          17           88                  2.0      0.19
NYG           J.McCarthy          15           74                  7.0      0.20
MIN              C.Wentz          18           70                  2.0

## 5. Ratings

In [38]:
def ratings(f, starters, w_other=W_OTHER_QB, k_off=K_OFF, k_def=K_DEF):
    lg = f["offensive_points"].mean()
    mine = f["qb"] == f["posteam"].map(starters)
    wt = np.where(mine, 1.0, w_other)
    g = f.assign(wt=wt, wpts=wt * f["offensive_points"], mine=mine.astype(int)).groupby("posteam")
    r = pd.DataFrame({"drives": g.size(), "off_ppd": g["offensive_points"].mean(),
                      "qb_share": g["mine"].mean(),
                      "off_rating": (g["wpts"].sum() + k_off * lg) / (g["wt"].sum() + k_off)})
    gd = f.groupby("defteam")
    r["drives_faced"] = gd.size()
    r["def_ppd"] = gd["offensive_points"].mean()
    r["def_rating"] = (gd["offensive_points"].sum() + k_def * lg) / (gd.size() + k_def)
    r["expected_starter"] = starters
    return r, lg

R, LEAGUE_PPD = ratings(f, starters)
print(f"league points per drive: {LEAGUE_PPD:.3f}")
print("off_ppd / def_ppd = raw.  off_rating / def_rating = what the projection uses (QB-weighted, pulled to average).")
print(R.sort_values("off_rating", ascending=False).round(2).to_string())

league points per drive: 2.234
off_ppd / def_ppd = raw.  off_rating / def_rating = what the projection uses (QB-weighted, pulled to average).
         drives  off_ppd  qb_share  off_rating  drives_faced  def_ppd  def_rating expected_starter
posteam                                                                                           
BUF          69     3.39      1.00        3.13            71     2.55        2.48          J.Allen
DAL          74     2.97      1.00        2.82            73     2.93        2.78       D.Prescott
SF           65     2.97      1.00        2.80            66     2.11        2.14          B.Purdy
CIN          67     2.96      1.00        2.79            67     2.24        2.24         J.Burrow
JAX          67     2.81      0.96        2.75            66     1.68        1.81       T.Lawrence
DET          71     2.87      1.00        2.73            69     2.65        2.56           J.Goff
CHI          81     2.64      0.64        2.72            81     1

## 6. Project the next week

In [39]:
def project_week(week, R, sched, coef=COEF):
    rows = []
    for g in sched[sched["week"] == week].itertuples():
        neutral = str(getattr(g, "location", "Home")) != "Home"
        out = {}
        for side, team, opp, h in [("home", g.home_team, g.away_team, 1), ("away", g.away_team, g.home_team, -1)]:
            if team not in R.index or opp not in R.index:
                raise ValueError(f"no rating for {team} or {opp}")
            out[side] = (coef["const"] + coef["off"] * R.loc[team, "off_rating"] + coef["deff"] * R.loc[opp, "def_rating"]
                         + coef["home"] * (0 if neutral else h) + coef["new_qb"] * (1 - R.loc[team, "qb_share"]))
        rows.append({"game_id": g.game_id, "week": week, "away": g.away_team, "home": g.home_team,
                     "away_qb": R.loc[g.away_team, "expected_starter"], "home_qb": R.loc[g.home_team, "expected_starter"],
                     "away_pts": out["away"], "home_pts": out["home"],
                     "home_margin": out["home"] - out["away"], "total": out["home"] + out["away"],
                     "already_played": bool(pd.notna(g.home_score)),
                     # market, for reference only. spread_line > 0 means the home team is favored.
                     "mkt_home_margin": getattr(g, "spread_line", np.nan), "mkt_total": getattr(g, "total_line", np.nan)})
    return pd.DataFrame(rows)

proj = project_week(WEEK, R, sched)
print(f"WEEK {WEEK} PROJECTIONS  (typical miss on one team's score: about {TYPICAL_MISS} points either way)")
print(proj.drop(columns=["game_id", "week", "already_played"]).round(1).to_string(index=False))
if proj["already_played"].any():
    print("\nNOTE:", int(proj["already_played"].sum()), "of these games already have a final score. They are saved with")
    print("already_played=True and the scorecard leaves them out, because they were not projected in advance.")

WEEK 9 PROJECTIONS  (typical miss on one team's score: about 7.5 points either way)
away home    away_qb    home_qb  away_pts  home_pts  home_margin  total  mkt_home_margin  mkt_total
 JAX  BAL T.Lawrence  J.Fagnano      25.9      19.0         -6.8   44.9             -8.5       40.5
 CIN  ATL   J.Burrow    M.Penix      27.0      23.3         -3.7   50.3             -3.0       52.5
 DEN  CAR      B.Nix    B.Young      21.9      24.9          3.1   46.8             -3.0       44.5
 DAL  IND D.Prescott    D.Jones      27.0      25.0         -1.9   52.0             -4.5       51.5
 NYJ   KC    G.Smith  P.Mahomes      22.1      26.4          4.3   48.5              9.5       47.5
 DET  MIA     J.Goff   M.Willis      27.2      24.1         -3.1   51.3             -6.5       49.5
 CLE   NO  S.Sanders   T.Shough      18.4      24.2          5.9   42.6              4.5       43.5
 NYG  PHI J.McCarthy   J.Milton      21.4      19.8         -1.6   41.2             -3.0       38.5
  LA  WAS M.Staf

## 7. Save, then grade against the real scores

Projections are saved **before** the games so they cannot be edited afterward. A game that already had a final score when it was saved is marked `already_played` and is not graded. Once scores are in, the scorecard compares this model to two baselines on exactly the same games: the betting market, and plain points for / points against. Design doc Sec 20: a model that cannot beat the naive baseline adds nothing.

In [40]:
out_path = os.path.join(FOLDER, f"hidden_value_projections_week{WEEK:02d}.csv")
if os.path.exists(out_path):
    print("already saved, NOT overwriting:", out_path)
    print("(delete the file by hand only if the games have not kicked off yet)")
else:
    proj.assign(saved_at=pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")).to_csv(out_path, index=False)
    print("saved", out_path)

already saved, NOT overwriting: C:\Users\brade\OneDrive - Worcester Polytechnic Institute (wpi.edu)\Desktop\Football\hidden_value_projections_week09.csv
(delete the file by hand only if the games have not kicked off yet)


In [41]:
def scorecard(folder, sched, include_already_played=False):
    files = sorted(glob.glob(os.path.join(folder, "hidden_value_projections_week*.csv")))
    if not files:
        print("no saved projections yet"); return None
    p = pd.concat([pd.read_csv(x) for x in files], ignore_index=True)
    s = sched[sched["home_score"].notna()]
    late = p["already_played"].astype(bool)
    if late.any() and not include_already_played:
        print(int(late.sum()), "saved game(s) skipped: they were saved after the final score was known")
        p = p[~late]
    m = p.merge(s[["game_id", "home_score", "away_score"]], on="game_id", how="inner")
    if m.empty:
        print("nothing to grade yet: no game that was projected in advance has a final score"); return None
    # naive baseline: average of a team's points per game and its opponent's points allowed per game,
    # using only games from earlier weeks
    def naive(team, opp, week):
        e = s[s["week"] < week]
        pf = pd.concat([e.loc[e["home_team"] == team, "home_score"], e.loc[e["away_team"] == team, "away_score"]]).mean()
        pa = pd.concat([e.loc[e["home_team"] == opp, "away_score"], e.loc[e["away_team"] == opp, "home_score"]]).mean()
        return (pf + pa) / 2
    rows = []
    for r in m.itertuples():
        for team, opp, pts, actual, mk in [
                (r.home, r.away, r.home_pts, r.home_score, r.mkt_total / 2 + r.mkt_home_margin / 2),
                (r.away, r.home, r.away_pts, r.away_score, r.mkt_total / 2 - r.mkt_home_margin / 2)]:
            rows.append({"week": r.week, "team": team, "opp": opp, "actual": actual, "model": pts,
                         "market": mk, "naive": naive(team, opp, r.week)})
    t = pd.DataFrame(rows)
    miss = t[["model", "market", "naive"]].sub(t["actual"], axis=0).abs()
    print(f"team-games graded: {len(t)}  (weeks {sorted(t['week'].unique().tolist())})")
    print("average miss per team, in points (lower is better):")
    print(miss.mean().round(2).to_string())
    win = np.sign(m["home_margin"]) == np.sign(m["home_score"] - m["away_score"])
    print(f"winner picked: {int(win.sum())} of {len(m)}")
    if len(t) < 100:
        print("NOTE: under ~100 team-games this comparison is mostly noise. Do not draw conclusions yet.")
    return t

graded = scorecard(FOLDER, sched)

29 saved game(s) skipped: they were saved after the final score was known
team-games graded: 84  (weeks [5, 6, 7, 8])
average miss per team, in points (lower is better):
model     7.78
market    7.29
naive     7.92
winner picked: 22 of 42
NOTE: under ~100 team-games this comparison is mostly noise. Do not draw conclusions yet.


## 8. Availability: what each offense has looked like under each QB

Descriptive, for the write-up and for sanity-checking Section 4. Uses the **research frame** (garbage time and clock drives excluded, design doc Sec 11 and 14). Every row shows its drive count. A split built on 8 drives is a case study, not a finding.

In [42]:
a = d[(d["garbage_time"] == 0) & (d["clock_expiration"] == 0)]
g = a.groupby(["posteam", "qb"])
by_qb = pd.DataFrame({"games": g["game_id"].nunique(), "drives": g.size(),
                      "pts_per_drive": g["offensive_points"].mean(),
                      "epa_per_play": g["epa_sum"].sum() / g["qualifying_plays"].sum(),
                      "success_rate": g["success_plays"].sum() / g["qualifying_plays"].sum()}).reset_index()
by_qb["share"] = by_qb["drives"] / by_qb.groupby("posteam")["drives"].transform("sum")
by_qb["expected_starter"] = by_qb["qb"] == by_qb["posteam"].map(starters)
multi = by_qb.groupby("posteam")["drives"].transform(lambda s: (s >= 5).sum()) >= 2
print("teams with two or more QBs who have each run 5+ drives:")
print(by_qb[multi & (by_qb["drives"] >= 5)].sort_values(["posteam", "drives"], ascending=[True, False]).round(3).to_string(index=False))

teams with two or more QBs who have each run 5+ drives:
posteam            qb  games  drives  pts_per_drive  epa_per_play  success_rate  share  expected_starter
    ATL       M.Penix      6      43          2.279        -0.043         0.442  0.683              True
    ATL        C.Rush      2      19          0.842        -0.353         0.393  0.302             False
    BAL     T.Huntley      4      33          1.818        -0.034         0.403  0.478             False
    BAL     L.Jackson      4      32          3.312         0.242         0.537  0.464             False
    CHI    C.Williams      5      42          3.000         0.186         0.481  0.677              True
    CHI      T.Bagent      3      14          1.929         0.002         0.524  0.226             False
    CHI      C.Keenum      1       6          3.333         0.111         0.489  0.097             False
    MIN      K.Murray      6      50          1.620        -0.154         0.371  0.758             False

### Same idea for any other offensive player (snap counts)

`missed_time()` lists regulars (50%+ of offensive snaps when they play) who have missed a team game, and uses the weekly roster status to say whether he was inactive / on a reserve list for it or was active and just did not play (a benched backup QB lands in the second group). `with_without()` splits the team's drives by games he played vs games he missed. The rule is identical for every team, so nobody is hand-picked.

In [43]:
try:
    snaps = to_pd(nfl.load_snap_counts(seasons=[SEASON]))
    snaps = snaps[snaps["game_type"] == "REG"].copy()
except Exception as e:
    snaps = None
    print("snap counts not available:", e)
try:
    rosters = to_pd(nfl.load_rosters_weekly(seasons=[SEASON]))
except Exception as e:
    rosters = None
    print("weekly rosters not available (missed games will not be labeled injured vs benched):", e)

def missed_time(snaps, rosters=None, min_avg_share=0.50, min_games_played=2):
    team_weeks = snaps.groupby("team")["week"].agg(lambda x: set(x))
    played = snaps[snaps["offense_snaps"] > 0]
    t = played.groupby(["pfr_player_id", "player", "team", "position"]).agg(
        games_played=("week", "nunique"), avg_share=("offense_pct", "mean"),
        weeks_played=("week", lambda x: set(x))).reset_index()
    t["weeks_missed"] = [sorted(team_weeks[tm] - wp) for tm, wp in zip(t["team"], t["weeks_played"])]
    t["games_missed"] = t["weeks_missed"].str.len()
    t = t[(t["avg_share"] >= min_avg_share) & (t["games_played"] >= min_games_played) & (t["games_missed"] > 0)].copy()
    if rosters is not None:
        # INA = inactive on game day, RES = a reserve list (injured reserve and similar).
        # A missed game with neither of those means he was healthy and simply did not play.
        off = rosters[rosters["status"].isin(["INA", "RES"])]
        off = set(zip(off["pfr_id"], off["week"]))
        t["weeks_inactive"] = [[w for w in wm if (pid, w) in off] for pid, wm in zip(t["pfr_player_id"], t["weeks_missed"])]
        known = t["pfr_player_id"].isin(set(rosters["pfr_id"].dropna()))
        t["why"] = np.where(~known, "unknown (no roster match)",
                            np.where(t["weeks_inactive"].str.len() > 0, "inactive / reserve", "active, did not play"))
    return t.drop(columns="weeks_played").sort_values(["games_missed", "avg_share"], ascending=False).reset_index(drop=True)

def with_without(a, snaps, pfr_player_id, team):
    s = snaps[(snaps["pfr_player_id"] == pfr_player_id) & (snaps["team"] == team) & (snaps["offense_snaps"] > 0)]
    x = a[a["posteam"] == team].copy()
    x["split"] = np.where(x["game_id"].isin(s["game_id"]), "with", "without")
    g = x.groupby("split")
    return pd.DataFrame({"games": g["game_id"].nunique(), "drives": g.size(),
                         "pts_per_drive": g["offensive_points"].mean(),
                         "epa_per_play": g["epa_sum"].sum() / g["qualifying_plays"].sum(),
                         "success_rate": g["success_plays"].sum() / g["qualifying_plays"].sum()})

if snaps is not None:
    mt = missed_time(snaps, rosters)
    skill = mt[mt["position"].isin(["QB", "RB", "WR", "TE"])]
    print(len(mt), "offensive regulars have missed at least one game. Skill positions:")
    print(skill.drop(columns=["pfr_player_id"]).round(2).to_string(index=False))
    # example: first skill-position player on the list. Change the two arguments to look at anyone.
    if len(skill):
        ex = skill.iloc[0]
        print(f"\n{ex['player']} ({ex['team']}): team offense with vs without him")
        print(with_without(a, snaps, ex["pfr_player_id"], ex["team"]).round(3).to_string())

182 offensive regulars have missed at least one game. Skill positions:
             player team position  games_played  avg_share          weeks_missed  games_missed     weeks_inactive                  why
        Cooper Rush  ATL       QB             2       0.79 [3, 4, 5, 6, 7, 8, 9]             7       [3, 4, 5, 6]   inactive / reserve
 Gardner Minshew II  ARI       QB             2       0.71 [1, 2, 3, 4, 5, 6, 7]             7                 [] active, did not play
       Tanner McKee  PHI       QB             3       1.00    [1, 2, 3, 4, 5, 9]             6          [1, 2, 3]   inactive / reserve
          Drew Lock  SEA       QB             3       0.97    [3, 4, 5, 6, 7, 8]             6             [7, 8]   inactive / reserve
      Dameon Pierce  PHI       RB             3       0.79    [1, 2, 3, 4, 8, 9]             6                [8]   inactive / reserve
         A.J. Brown   NE       WR             3       0.70    [2, 3, 4, 5, 8, 9]             6 [2, 3, 4, 5, 8, 9]   ina

## 9. Appendix: where the constants come from (slow, optional)

Rebuilds the drive table for 2021-2025 with the same rules as Phase 2 (only the columns the projection needs), then, for every week 4-18, rates each team on prior weeks only and fits the points formula. Each season is predicted from the other four. Takes several minutes and pulls five seasons of play-by-play. Set `RUN_BACKTEST = True` to run it.

In [44]:
RUN_BACKTEST = False
BT_SEASONS = [2021, 2022, 2023, 2024, 2025]

def build_drives_min(pbp):
    """Phase 2 rules, forecasting columns only: one row per qualifying offensive drive."""
    p = pbp[(pbp["season_type"] == "REG") & pbp["fixed_drive"].notna()].copy()
    p["drive_id"] = p["game_id"].astype(str) + "_" + p["fixed_drive"].astype(int).astype(str)
    qual = p["play_type"].fillna("").isin(["pass", "run"]) & (p["qb_kneel"].fillna(0) != 1) & (p["qb_spike"].fillna(0) != 1)
    p["is_qual"] = qual
    p["posteam_q"] = np.where(qual, p["posteam"], np.nan)
    p["defteam_q"] = np.where(qual, p["defteam"], np.nan)
    g = p.groupby("drive_id", sort=False)
    out = g.agg(game_id=("game_id", "first"), season=("season", "first"), week=("week", "first"),
                posteam=("posteam_q", "first"), defteam=("defteam_q", "first"), fixed_drive=("fixed_drive", "first"),
                drive_result=("fixed_drive_result", "first"), qualifying_plays=("is_qual", "sum")).reset_index()
    out = out[(out["qualifying_plays"] > 0) & out["posteam"].notna()].copy()
    res = out["drive_result"].fillna("")
    out["offensive_points"] = np.where(res == "Touchdown", 7, np.where(res == "Field goal", 3, 0))
    out["clock_expiration"] = res.str.startswith("End").astype(int)
    q = drive_qb(pbp[pbp["season_type"] == "REG"])
    out = out.join(q, on="drive_id").sort_values(["game_id", "posteam", "fixed_drive"])
    for c in ["qb_id", "qb"]:
        out[c] = out.groupby(["game_id", "posteam"])[c].ffill()
        out[c] = out.groupby(["game_id", "posteam"])[c].bfill()
    return out[out["clock_expiration"] == 0]

def game_starters(pbp):
    """QB on each team's first dropback of each game = the starter you would have known before kickoff."""
    p = pbp[(pbp["season_type"] == "REG") & (pbp["qb_dropback"] == 1) & pbp["id"].notna() & pbp["posteam"].notna()]
    return p.sort_values(["game_id", "play_id"]).drop_duplicates(["game_id", "posteam"]).set_index(["game_id", "posteam"])["name"]

def backtest_rows(fr, st, sc, w_other, k_off, k_def, weeks=range(4, 19)):
    """One row per team-game: ratings from prior weeks only, plus the actual score."""
    rows = []
    for s in sorted(fr["season"].unique()):
        fs, ss = fr[fr["season"] == s], sc[(sc["season"] == s) & sc["home_score"].notna()]
        for w in weeks:
            past, games = fs[fs["week"] < w], ss[ss["week"] == w]
            if games.empty or past.empty:
                continue
            start = {}
            for gm in games.itertuples():
                for team in (gm.home_team, gm.away_team):
                    start[team] = st.get((gm.game_id, team), None)
            r, _ = ratings(past, pd.Series(start), w_other, k_off, k_def)
            e = ss[ss["week"] < w]
            pf = pd.concat([e.set_index("home_team")["home_score"], e.set_index("away_team")["away_score"]]).groupby(level=0).mean()
            pa = pd.concat([e.set_index("home_team")["away_score"], e.set_index("away_team")["home_score"]]).groupby(level=0).mean()
            for gm in games.itertuples():
                neutral = gm.location != "Home"
                for team, opp, pts, h, mk in [
                        (gm.home_team, gm.away_team, gm.home_score, 1, gm.total_line / 2 + gm.spread_line / 2),
                        (gm.away_team, gm.home_team, gm.away_score, -1, gm.total_line / 2 - gm.spread_line / 2)]:
                    if team not in r.index or opp not in r.index:
                        continue
                    rows.append({"season": s, "week": w, "game_id": gm.game_id, "team": team, "pts": pts,
                                 "off": r.loc[team, "off_rating"], "deff": r.loc[opp, "def_rating"],
                                 "home": 0 if neutral else h, "new_qb": 1 - r.loc[team, "qb_share"],
                                 "pf": pf.get(team, np.nan), "pa": pa.get(opp, np.nan), "market": mk,
                                 "league": pd.concat([e["home_score"], e["away_score"]]).mean()})
    return pd.DataFrame(rows)

def fit(X, cols):
    A = np.column_stack([np.ones(len(X))] + [X[c].values for c in cols])
    coef, *_ = np.linalg.lstsq(A, X["pts"].values, rcond=None)
    return coef

def loso(X, cols):
    """Leave-one-season-out predictions."""
    pred = pd.Series(np.nan, index=X.index)
    for s in X["season"].unique():
        tr, te = X[X["season"] != s], X[X["season"] == s]
        c = fit(tr, cols)
        pred[te.index] = np.column_stack([np.ones(len(te))] + [te[k].values for k in cols]) @ c
    return pred

if RUN_BACKTEST:
    frames, starts = [], {}
    for y in BT_SEASONS:
        hp = to_pd(nfl.load_pbp(seasons=[y]), PBP_COLS)
        frames.append(build_drives_min(hp))
        starts.update(game_starters(hp).to_dict())
        print("built", y, len(frames[-1]), "drives")
    fr = pd.concat(frames, ignore_index=True)
    sc = to_pd(nfl.load_schedules(seasons=BT_SEASONS)); sc = sc[sc["game_type"] == "REG"]

    X = backtest_rows(fr, starts, sc, W_OTHER_QB, K_OFF, K_DEF).dropna(subset=["pts", "off", "deff", "pf", "pa"]).reset_index(drop=True)
    X0 = backtest_rows(fr, starts, sc, 1.0, K_OFF, K_DEF).dropna(subset=["pts", "off", "deff", "pf", "pa"]).reset_index(drop=True)
    assert len(X0) == len(X) and (X0["game_id"] == X["game_id"]).all() and (X0["team"] == X["team"]).all()
    cols = ["off", "deff", "home", "new_qb"]
    preds = {"league average": X["league"], "points for / against": loso(X, ["pf", "pa", "home"]),
             "drive model, no QB info": loso(X0, ["off", "deff", "home"]).set_axis(X.index),
             "this notebook": loso(X, cols), "betting market": X["market"]}
    print(f"\n{len(X)} team-games, weeks 4-18, {BT_SEASONS[0]}-{BT_SEASONS[-1]}. Average miss per team (points):")
    for k, v in preds.items():
        print(f"   {k:26s} {(v - X['pts']).abs().mean():.2f}")
    print("\nwinner picked (games with both teams projected):")
    for k, v in preds.items():
        if k == "league average":
            continue
        gm = X.assign(p=v).sort_values("home").groupby("game_id").agg(pa=("p", "first"), ph=("p", "last"),
                                                                    a=("pts", "first"), h=("pts", "last"), n=("p", "size"))
        gm = gm[(gm["n"] == 2) & (gm["a"] != gm["h"])]
        print(f"   {k:26s} {(np.sign(gm['ph'] - gm['pa']) == np.sign(gm['h'] - gm['a'])).mean():.1%}")
    low = X["new_qb"] > 0.7
    print(f"\ngames where the starter had run under 30% of his team's drives: {int(low.sum())}")
    print(f"   average over-projection without QB info: {(preds['drive model, no QB info'] - X['pts'])[low].mean():+.2f} pts")
    print(f"   average over-projection with QB info:    {(preds['this notebook'] - X['pts'])[low].mean():+.2f} pts")
    c = fit(X, cols)
    print("\nCOEF fitted on all seasons (paste into Section 2 if you re-derive):")
    print({k: round(float(v), 2) for k, v in zip(["const", "off", "deff", "home", "new_qb"], c)})